In [ ]:
import logging
import sklearn
from sklearn.metrics import pairwise_distances
from sentence_transformers import CrossEncoder

logger = logging.getLogger(__name__)


In [ ]:
class Reranker:
    def __init__(
        self,
        model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
        device=None,
    ):
        model_kwargs = {}
        if device is not None:
            model_kwargs["device"] = device

        self.model = CrossEncoder(model_name, **model_kwargs)

    def rerank(self, query, documents, top_n=5):
        """Rerank retrieved LangChain Documents by relevance."""

        if not query or not query.strip():
            raise ValueError("Query cannot be empty")

        if not documents:
            return []

        pairs = [
            [query, doc.page_content]
            for doc in documents
        ]

        scores = self.model.predict(pairs)

        ranked = sorted(
            zip(documents, scores),
            key=lambda item: float(item[1]),
            reverse=True,
        )

        results = []

        for doc, score in ranked[:top_n]:
            results.append({
                "document": doc,
                "score": float(score),
            })

        return results